##### Import libraries

In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

### Data Preprocessing

##### Data loading and cleaning

In [ ]:
df = pd.read_csv('../dataset/raw/oasis_longitudinal.csv')

# 1. Map target 'Group' to binary (0 = Nondemented, 1 = Demented/Converted)
df['Group'] = df['Group'].replace(['Converted'], ['Demented'])
df['Group'] = df['Group'].map({'Nondemented': 0, 'Demented': 1})

# 2. Encode Gender ('M/F') to binary (Male = 1, Female = 0)
df['M/F'] = df['M/F'].map({'M': 1, 'F': 0})

# 3. Drop non-predictive columns
# 'Subject ID' and 'MRI ID' are just identifiers. 
# 'Hand' is dropped because 100% of the subjects are right-handed (zero variance).
# 'Visit' and 'MR Delay' are dropped to prevent data leakage and evaluate purely on baseline brain volume/demographics.
df = df.drop(['Subject ID', 'MRI ID', 'Hand', 'Visit', 'MR Delay'], axis=1)

print("Dataset shape after cleaning:", df.shape)
print("\nMissing values before imputation:\n", df.isnull().sum())

Dataset shape after cleaning: (373, 10)

Missing values before imputation:
 Group     0
M/F       0
Age       0
EDUC      0
SES      19
MMSE      2
CDR       0
eTIV      0
nWBV      0
ASF       0
dtype: int64


##### Data splitting

In [15]:
# define features and target
X = df.drop('Group', axis=1)
Y = df['Group']

# data splitting
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42, stratify=Y)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Testing set size: {X_test.shape[0]} samples")

Training set size: 298 samples
Testing set size: 75 samples


##### Feature scaling & handling missing values

In [27]:
# 1. Impute missing values with the median of the training set
imputer = SimpleImputer(strategy='median')
X_train_imputed = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns)
# don't fit (learn) from the testing set
X_test_imputed = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns)

# 2. Scale the numerical features
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_imputed), columns=X_train_imputed.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test_imputed), columns=X_test_imputed.columns)

print("Scaling and Imputation complete.")
X_train_imputed.isnull().sum()

Scaling and Imputation complete.


M/F     0
Age     0
EDUC    0
SES     0
MMSE    0
CDR     0
eTIV    0
nWBV    0
ASF     0
dtype: int64

##### Applying SMOTE to handle class imbalance

In [28]:
# Check class balance before SMOTE (6 extra nondemented subjects in the training set)
print("Class distribution before SMOTE:")
print(Y_train.value_counts())

# Apply SMOTE to the training set only
smote = SMOTE(random_state=42)
X_train_resampled, Y_train_resampled = smote.fit_resample(X_train_scaled, Y_train)

# Check class balance after SMOTE
print("\nClass distribution after SMOTE:")
print(Y_train_resampled.value_counts())

Class distribution before SMOTE:
Group
0    152
1    146
Name: count, dtype: int64

Class distribution after SMOTE:
Group
1    152
0    152
Name: count, dtype: int64


##### Export processed data

In [30]:
# Save the final preprocessed datasets to the 'processed' folder
X_train_resampled.to_csv('../dataset/processed/X_train.csv', index=False)
X_test_scaled.to_csv('../dataset/processed/X_test.csv', index=False)
Y_train_resampled.to_csv('../dataset/processed/Y_train.csv', index=False)
Y_test.to_csv('../dataset/processed/Y_test.csv', index=False)

print("Preprocessed data successfully saved to the 'dataset/processed' directory!")

Preprocessed data successfully saved to the 'dataset/processed' directory!
